In [1]:
import pandas as pd
import mysql.connector
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Connexion à la base de données observatoire_velov
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="observatoire_velov"
)

In [ ]:
# 1. Extraction d'un échantillon propre (id et velos à la place de station_id et vélos_disponibles)
query = """
SELECT date_enregistrement, velos 
FROM stations_historique 
WHERE id = 9012  -- Utilise le nom de colonne 'id' correct
ORDER BY date_enregistrement ASC
LIMIT 10000;
"""

In [ ]:
# Tracer l'ACF et le PACF
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
plot_acf(df['vélos_disponibles'], ax=ax1, lags=48) # On regarde 48h en arrière
plot_pacf(df['vélos_disponibles'], ax=ax2, lags=48)
plt.show()

In [ ]:
# Chargement dans un DataFrame pandas
df = pd.read_sql(query, connection)
df['date_enregistrement'] = pd.to_datetime(df['date_enregistrement'])
df.set_index('date_enregistrement', inplace=True)
print(df.head())

# Important : ARIMA a besoin d'une fréquence régulière (ex: chaque heure)
# Si tes données sont irrégulières, on fait un rééchantillonnage
df = df.resample('H').mean().fillna(method='ffill')

In [ ]:
# Configuration du modèle (p, d, q) et de la saisonnalité (P, D, Q, S)
# S = 24 pour un cycle journalier (données horaires)
model = SARIMAX(df['vélos_disponibles'], 
                order=(1, 1, 1), 
                seasonal_order=(1, 1, 1, 24))

results = model.fit()